# ⚙️ openBIS Chatbot Setup

Configure your LLM provider and credentials for the **NanoSurfaces Prime** openBIS assistant. The chatbot operates in **strict read-only mode** to protect your experimental research data.

In [ ]:
%%capture
import os
import json
import time
from pathlib import Path
from IPython.display import display, HTML, Javascript
import ipywidgets as ipw
from src import utils


In [ ]:
# Load base configuration
CONFIG_PATH = "config/config.json"
LLM_CONFIG_PATH = "/home/jovyan/api_keys/llm_config.json"
CONFIG = utils.read_json(CONFIG_PATH) if Path(CONFIG_PATH).exists() else {}
increase_buttons_size = HTML(data="".join(CONFIG.get("save_home_buttons_settings", [])))

# Load existing saved configuration if available
SAVED_CFG = {}
if Path(LLM_CONFIG_PATH).exists():
    try:
        SAVED_CFG = utils.read_json(LLM_CONFIG_PATH)
    except Exception:
        SAVED_CFG = {}

# Card CSS styling
style_html = HTML('''
<style>
  .container, #notebook-container {
    width: 98% !important;
    max-width: 1200px !important;
    margin-left: auto !important;
    margin-right: auto !important;
  }
  .setup-card {
    background: #ffffff;
    border: 1px solid #e1e4e8;
    border-radius: 10px;
    padding: 24px;
    margin-bottom: 20px;
    box-shadow: 0 2px 8px rgba(0, 0, 0, 0.05);
    max-width: 860px;
    margin: 0 auto;
  }
  .status-badge-ok {
    background: #e6f4ea;
    color: #137333;
    padding: 6px 14px;
    border-radius: 20px;
    font-weight: 600;
    font-size: 13px;
    display: inline-block;
  }
  .status-badge-warn {
    background: #fef7e0;
    color: #b06000;
    padding: 6px 14px;
    border-radius: 20px;
    font-weight: 600;
    font-size: 13px;
    display: inline-block;
  }
  .section-title {
    font-size: 16px;
    font-weight: bold;
    color: #1f2937;
    margin-bottom: 12px;
    border-bottom: 2px solid #f3f4f6;
    padding-bottom: 6px;
  }
</style>
''')

# Header status widget
def render_status_html():
    if SAVED_CFG.get("llm_api_key") and SAVED_CFG.get("llm_provider"):
        prov = SAVED_CFG.get("llm_provider")
        mod = SAVED_CFG.get("llm_model", "")
        return f'''
        <div style="margin-bottom: 15px;">
          <span class="status-badge-ok">● Configured: <b>{prov}</b> ({mod})</span>
        </div>
        '''
    return '''
    <div style="margin-bottom: 15px;">
      <span class="status-badge-warn">○ Not Configured (API key required)</span>
    </div>
    '''

status_box = ipw.HTML(value=render_status_html())

# Model choices mapping
MODEL_OPTIONS = {
    "CSCS": [
        "google/gemma-4-31B-it",
        "swiss-ai/Apertus-70B-Instruct-2509",
        "swiss-ai/Apertus-8B-Instruct-2509",
        "swiss-ai/Apertus-v1.5-70B",
        "swiss-ai/Apertus-v1.5-70B-thinking",
        "swiss-ai/Apertus-v1.5-8B",
        "moonshotai/Kimi-K2.7-Code",
        "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16",
        "zai-org/GLM-5.3",
        "zai-org/GLM-5.2",
    ],
    "Google Gemini": [
        "gemini-2.5-flash",
        "gemini-2.5-pro",
        "gemini-2.0-flash",
        "gemini-2.0-flash-lite",
    ],
    "OpenAI": [
        "gpt-4o",
        "gpt-4o-mini",
        "gpt-4-turbo",
    ],
}

# UI Elements
saved_prov = SAVED_CFG.get("llm_provider", "CSCS")
if saved_prov not in MODEL_OPTIONS:
    saved_prov = "CSCS"

provider_dropdown = ipw.Dropdown(
    options=["CSCS", "Google Gemini", "OpenAI"],
    value=saved_prov,
    description="Provider:",
    style={"description_width": "140px"},
    layout=ipw.Layout(width="550px"),
)

saved_model = SAVED_CFG.get("llm_model", "")
models_for_prov = MODEL_OPTIONS.get(saved_prov, [])
init_model = saved_model if saved_model in models_for_prov else (models_for_prov[0] if models_for_prov else "")

model_dropdown = ipw.Dropdown(
    options=models_for_prov,
    value=init_model,
    description="LLM Model:",
    style={"description_width": "140px"},
    layout=ipw.Layout(width="550px"),
)

api_key_input = ipw.Password(
    value=SAVED_CFG.get("llm_api_key", ""),
    placeholder="Paste your API key here",
    description="API Key:",
    style={"description_width": "140px"},
    layout=ipw.Layout(width="550px"),
)

# Test Connection button & status
test_conn_button = ipw.Button(
    description="Test Connection",
    icon="check-circle",
    button_style="info",
    layout=ipw.Layout(width="160px", height="36px"),
)
test_status_label = ipw.HTML(value="")
test_hbox = ipw.HBox([test_conn_button, test_status_label], layout=ipw.Layout(align_items="center", margin="10px 0 10px 140px"))

# Langfuse Observability Section
langfuse_checkbox = ipw.Checkbox(
    value=bool(SAVED_CFG.get("langfuse_host")),
    description="Enable Langfuse Tracing (Optional)",
    style={"description_width": "140px"},
    indent=False,
)
langfuse_host_input = ipw.Text(
    value=SAVED_CFG.get("langfuse_host", ""),
    placeholder="https://cloud.langfuse.com",
    description="Langfuse Host:",
    style={"description_width": "140px"},
    layout=ipw.Layout(width="550px"),
)
langfuse_pkey_input = ipw.Text(
    value=SAVED_CFG.get("langfuse_pkey", ""),
    placeholder="pk-lf-...",
    description="Public Key:",
    style={"description_width": "140px"},
    layout=ipw.Layout(width="550px"),
)
langfuse_skey_input = ipw.Password(
    value=SAVED_CFG.get("langfuse_skey", ""),
    placeholder="sk-lf-...",
    description="Secret Key:",
    style={"description_width": "140px"},
    layout=ipw.Layout(width="550px"),
)
langfuse_vbox = ipw.VBox(
    [langfuse_host_input, langfuse_pkey_input, langfuse_skey_input],
    layout=ipw.Layout(display="flex" if langfuse_checkbox.value else "none"),
)

# Action buttons
save_button = ipw.Button(
    description="Save Settings",
    icon="save",
    button_style="success",
    layout=ipw.Layout(width="150px", height="42px"),
)
home_button = ipw.Button(
    description="Main Menu",
    icon="home",
    layout=ipw.Layout(width="130px", height="42px"),
)
notification_html = ipw.HTML(value="")

# Callbacks
def on_provider_change(change):
    prov = change["new"]
    opts = MODEL_OPTIONS.get(prov, [])
    model_dropdown.options = opts
    if opts:
        model_dropdown.value = opts[0]
    test_status_label.value = ""

def on_langfuse_toggle(change):
    langfuse_vbox.layout.display = "flex" if change["new"] else "none"

def test_connection(b):
    key = api_key_input.value.strip()
    prov = provider_dropdown.value
    mod = model_dropdown.value
    if not key:
        test_status_label.value = "<span style='color: #d93025; font-weight: bold; margin-left: 10px;'>⚠️ Please enter an API key first.</span>"
        return
    test_status_label.value = "<span style='color: #1a73e8; margin-left: 10px;'>⏳ Testing connection to provider...</span>"
    try:
        t0 = time.time()
        if prov == "CSCS":
            from openai import OpenAI
            client = OpenAI(base_url="https://api.inference.cscs.ch/v1", api_key=key, timeout=10.0)
            resp = client.chat.completions.create(
                model=mod,
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=2,
            )
        elif prov == "OpenAI":
            from openai import OpenAI
            client = OpenAI(api_key=key, timeout=10.0)
            resp = client.chat.completions.create(
                model=mod,
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=2,
            )
        elif prov == "Google Gemini":
            from google import genai
            client = genai.Client(api_key=key)
            resp = client.models.generate_content(model=mod, contents="ping")
        latency = round(time.time() - t0, 2)
        test_status_label.value = f"<span style='color: #188038; font-weight: bold; margin-left: 10px;'>✅ Success ({latency}s)</span>"
    except Exception as e:
        test_status_label.value = f"<span style='color: #d93025; font-weight: bold; margin-left: 10px;'>❌ Failed: {str(e)[:120]}</span>"

def save_settings(b):
    key = api_key_input.value.strip()
    if not key:
        notification_html.value = "<div style='color: #d93025; font-weight: bold; margin-top: 10px;'>⚠️ Please enter your LLM API key before saving.</div>"
        return
    new_cfg = {
        "llm_provider": provider_dropdown.value,
        "llm_model": model_dropdown.value,
        "llm_api_key": key,
        "langfuse_host": langfuse_host_input.value.strip() if langfuse_checkbox.value else "",
        "langfuse_pkey": langfuse_pkey_input.value.strip() if langfuse_checkbox.value else "",
        "langfuse_skey": langfuse_skey_input.value.strip() if langfuse_checkbox.value else "",
    }
    os.makedirs(os.path.dirname(LLM_CONFIG_PATH), exist_ok=True)
    utils.write_json(new_cfg, LLM_CONFIG_PATH)
    SAVED_CFG.clear()
    SAVED_CFG.update(new_cfg)
    status_box.value = render_status_html()
    notification_html.value = "<div style='color: #188038; font-weight: bold; margin-top: 10px;'>✅ Configuration saved successfully!</div>"

def go_home(b):
    display(Javascript(data='window.location.replace("0_home.ipynb")'))

provider_dropdown.observe(on_provider_change, names="value")
langfuse_checkbox.observe(on_langfuse_toggle, names="value")
test_conn_button.on_click(test_connection)
save_button.on_click(save_settings)
home_button.on_click(go_home)

# Main Layout Panel
card = ipw.VBox([
    status_box,
    ipw.HTML("<div class='section-title'>LLM Model & Provider</div>"),
    provider_dropdown,
    model_dropdown,
    api_key_input,
    test_hbox,
    ipw.HTML("<div class='section-title' style='margin-top: 15px;'>Observability & Tracing</div>"),
    langfuse_checkbox,
    langfuse_vbox,
    ipw.HTML("<div style='margin-top: 20px;'></div>"),
    ipw.HBox([save_button, home_button], layout=ipw.Layout(gap="15px")),
    notification_html,
], layout=ipw.Layout(padding="24px", border="1px solid #e1e4e8", border_radius="10px", width="96%", max_width="860px", margin="0 auto"))

display(style_html)
display(increase_buttons_size)
display(card)
